# Notebook 02 — Historical Replication: CLMX Variance Decomposition

**Project Parallax | Phase 0 / Phase 1 Bridge | v0.1.3**

---

## Purpose

This notebook implements the three-component variance decomposition from:

> Campbell, J. Y., Lettau, M., Malkiel, B. G., & Xu, Y. (2001). *Have Individual Stocks Become More Volatile? An Empirical Exploration of Idiosyncratic Risk.* Journal of Finance, 56(1), 1–43.

Extended and re-confirmed in:

> Campbell, J. Y., Lettau, M., Malkiel, B. G., & Xu, Y. (2022). *Idiosyncratic Equity Risk Two Decades Later.* NBER Working Paper No. 29916.

**Goal:** Implement the methodology faithfully using public data, produce MKT/IND/FIRM time series for S&P 500 (2010–2024), and compare directional behavior to CLMX (2022) Figures 2–4.

**This notebook does not test H1.** It establishes whether the replication framework behaves consistently with the literature before any Parallax-specific hypothesis testing begins.

---

## Structure — Learning-First Architecture

Each step explains before it computes. Functions appear in Step 12 only — they are *not* defined early and called silently.

| Step | Topic | Type |
|---|---|---|
| 1 | Mathematical framework + synthetic worked example | Explanation + manual calculation |
| 2 | Raw data inspection | Diagnostic |
| 3 | Universe and FF49 classification validation | Data infrastructure + validation |
| 4 | Weight construction | Data infrastructure + diagnostics |
| 5 | Manual decomposition — MKT only | Core implementation (partial) |
| 6 | Manual decomposition — IND only | Core implementation (partial) |
| 7 | Manual decomposition — FIRM only | Core implementation (partial) |
| 8 | Full-period monthly accumulation | Core implementation (full loop) |
| 9 | Reconciliation and sanity checks | Validation |
| 10 | Benchmark comparison to CLMX (2022) | Analysis |
| 11 | Limitations | Methodology |
| 12 | Reusable functions | Reference |

---

**Build constraint:** Work through Steps 5–7 manually before running Step 8. The decomposition must be understood at the single-month level before it is applied at scale.

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import yfinance as yf
import requests
import io
import zipfile
import time
import warnings
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from pathlib import Path
from typing import Dict, Optional, Tuple

warnings.filterwarnings('ignore')

# ── Project paths ─────────────────────────────────────────────────────────────
PROJECT_ROOT = Path('../')
DATA_DIR     = PROJECT_ROOT / 'data'
DATA_DIR.mkdir(exist_ok=True)

# ── Study window ──────────────────────────────────────────────────────────────
# Decision D-020: Primary window 2010–2024.
# May extend to 2005–2024 pending Phase 1 coverage diagnostics (Step 2).
STUDY_START = '2010-01-01'
STUDY_END   = '2024-12-31'

print(f'Study window : {STUDY_START} → {STUDY_END}')
print(f'Data cache   : {DATA_DIR.resolve()}')

---

## Step 1 — Mathematical Framework and Synthetic Worked Example

### 1.1 The Return Decomposition

CLMX decompose each stock's daily return into three orthogonal components:

$$r_{i,j,t,d} = \underbrace{\mu_{t,d}}_{\text{market}} + \underbrace{\eta_{j,t,d}}_{\text{industry excess market}} + \underbrace{\varepsilon_{i,j,t,d}}_{\text{firm excess industry}}$$

Where:
- $\mu_{t,d}$ = value-weighted market return on day $d$ of month $t$
- $\eta_{j,t,d} = r_{j,t,d} - \mu_{t,d}$ = VW industry $j$ return minus market return
- $\varepsilon_{i,j,t,d} = r_{i,t,d} - r_{j,t,d}$ = firm return minus its industry return

The components are orthogonal by construction — each is an excess over the prior level.

### 1.2 Monthly Variance Accumulation

$$\text{MKT}_t = \sum_{d=1}^{D_t} \mu_{t,d}^2 \qquad \text{IND}_t = \sum_j W_{j,t} \cdot \sum_{d=1}^{D_t} \eta_{j,t,d}^2 \qquad \text{FIRM}_t = \sum_j W_{j,t} \cdot \sum_{i \in j} w_{i,j,t} \cdot \sum_{d=1}^{D_t} \varepsilon_{i,j,t,d}^2$$

**Estimator confirmed** — CLMX (2022) NBER WP 29916, Figure notes 1–4:
1. **Raw squared returns** — daily return components are NOT demeaned before squaring
2. **Not normalized by trading-day count** — monthly sum left as raw accumulation
3. **Value-weighted primary** — $W_{j,t}$ is industry share of total market cap

### 1.3 Annual Aggregation

$$\text{MKT}_{\text{year}} = \sum_{t \in \text{year}} \text{MKT}_t \qquad \text{Volatility} = \sqrt{\text{Annual Variance}}$$

### 1.4 What Each Component Measures

| Component | What it captures | Expected behavior |
|---|---|---|
| MKT | Variance common to all stocks | Spikes during crises |
| IND | Within-industry variance, orthogonal to market | Sector rotation and industry shocks |
| FIRM | Variance specific to individual companies after market and industry are removed | The "idiosyncratic" component |

**Important:** CLMX FIRM variance is NOT factor-controlled. It removes market and industry components only. It is conceptually different from the factor-model residual variance that will be computed in Phase 5 (Layer 5 of the analytical framework). See docs/methodology.md for the full distinction.

In [ ]:
# ── Synthetic data: 3 stocks, 2 industries, 5 trading days ───────────────────
synth_returns = pd.DataFrame({
    'TECH_1': [ 0.020,  0.010, -0.005,  0.015,  0.003],
    'TECH_2': [ 0.012,  0.008, -0.010,  0.020, -0.002],
    'ENRG_1': [-0.005,  0.003,  0.008, -0.003,  0.010],
})
synth_industry = {'TECH_1': 'Technology', 'TECH_2': 'Technology', 'ENRG_1': 'Energy'}
synth_mkt_wts  = pd.Series({'TECH_1': 0.40, 'TECH_2': 0.30, 'ENRG_1': 0.30})

print('Daily returns:')
print(synth_returns)
print(f'\nMarket weights: {synth_mkt_wts.to_dict()}')
print(f'Industry map:   {synth_industry}')

In [ ]:
# ── Step 1a: VW market return (μ_d) ──────────────────────────────────────────
synth_mu = synth_returns.mul(synth_mkt_wts, axis='columns').sum(axis='columns')
MKT_synth = (synth_mu**2).sum()

print('Daily market return μ_d:')
print(synth_mu.round(6))
print(f'\nMKT = Σ μ_d² = {MKT_synth:.8f}')

In [ ]:
# ── Step 1b: Industry components (IND) ───────────────────────────────────────
synth_ind_groups = {}
for ind in set(synth_industry.values()):
    members = [t for t, i in synth_industry.items() if i == ind]
    ind_wts = synth_mkt_wts[members]
    W_j     = ind_wts.sum()
    w_ij    = ind_wts / W_j
    r_j     = synth_returns[members].mul(w_ij, axis='columns').sum(axis='columns')
    eta_j   = r_j - synth_mu
    synth_ind_groups[ind] = {'members': members, 'W_j': W_j, 'w_ij': w_ij,
                              'r_j': r_j, 'eta': eta_j}
    print(f'{ind}: W_j={W_j:.2f}, η_j,d = {eta_j.round(6).to_dict()}')

IND_synth = sum(d['W_j'] * (d['eta']**2).sum() for d in synth_ind_groups.values())
print(f'\nIND = Σ_j W_j · Σ_d η² = {IND_synth:.8f}')

In [ ]:
# ── Step 1c: Firm components (FIRM) + reconciliation ─────────────────────────
FIRM_synth = 0.0
for ind, d in synth_ind_groups.items():
    for ticker in d['members']:
        eps = synth_returns[ticker] - d['r_j']
        FIRM_synth += d['W_j'] * d['w_ij'][ticker] * (eps**2).sum()

total_synth   = MKT_synth + IND_synth + FIRM_synth
vw_total      = ((synth_returns**2).sum() * synth_mkt_wts).sum()

print('═' * 50)
print('Synthetic Worked Example — Reconciliation')
print('═' * 50)
print(f'MKT : {MKT_synth:.8f}  ({MKT_synth/total_synth:.1%})')
print(f'IND : {IND_synth:.8f}  ({IND_synth/total_synth:.1%})')
print(f'FIRM: {FIRM_synth:.8f}  ({FIRM_synth/total_synth:.1%})')
print(f'Sum : {total_synth:.8f}')
print(f'VW-avg individual variance: {vw_total:.8f}')
print(f'Difference (cross-product): {vw_total - total_synth:.2e}')
print()
print('If difference is < 1% of total, implementation logic is correct.')
print('Differences are expected cross-product terms — not a computational error.')

---

## Step 2 — Raw Data Inspection

Before building any decomposition, inspect the data for:
- Shape and date coverage
- Missing-data patterns (are gaps clustered in certain tickers or periods?)
- Return distribution anomalies (extreme single-day returns that may indicate data errors)
- Coverage by study sub-period (supports D-020 start-date decision)

This step runs after data is fetched in Step 3 uses prices loaded there. Cells are ordered for execution: run Step 3 data-loading cells first, then return to Step 2 inspection cells.

**Anomaly threshold:** Single-day returns > 50% or < -50% are flagged. These may be data errors (bad price), corporate events (spinoffs, ADR conversions), or genuine extreme events. Each flagged observation requires a judgment call — do not drop automatically.

In [ ]:
# ── Run after Step 3 data loading ────────────────────────────────────────────
# This cell references `daily_returns` computed in Step 3.
# Execute Step 3 fetch cells first, then run this block.

try:
    dr = daily_returns  # noqa: F821 — defined in Step 3
except NameError:
    print('daily_returns not yet defined. Run Step 3 data-loading cells first.')
    raise

print('═' * 55)
print('Raw data inspection')
print('═' * 55)
print(f'Shape            : {dr.shape[0]:,} days × {dr.shape[1]:,} tickers')
print(f'Date range       : {dr.index[0].date()} → {dr.index[-1].date()}')
print(f'Missing cells    : {dr.isna().sum().sum():,} ({dr.isna().mean().mean():.1%})')
print()

# Missing rate by year
missing_by_year = dr.isna().groupby(dr.index.year).mean().mean(axis=1)
print('Missing rate by year:')
for yr, rate in missing_by_year.items():
    bar = '█' * int(rate * 200)
    print(f'  {yr}: {rate:.1%}  {bar}')

print()

# Tickers with highest missing rates
ticker_missing = dr.isna().mean().sort_values(ascending=False)
print(f'Tickers with > 10% missing data: {(ticker_missing > 0.10).sum()}')
print('Top 10 by missing rate:')
print(ticker_missing.head(10).map('{:.1%}'.format).to_string())

In [ ]:
# ── Anomaly screening: extreme single-day returns ─────────────────────────────
ANOMALY_THRESHOLD = 0.50  # 50% single-day move

extremes = dr[(dr.abs() > ANOMALY_THRESHOLD)].stack()
print(f'Single-day moves > {ANOMALY_THRESHOLD:.0%}: {len(extremes)} observations')

if len(extremes) > 0:
    extremes = extremes.reset_index()
    extremes.columns = ['date', 'ticker', 'return']
    extremes = extremes.sort_values('return', key=abs, ascending=False)
    print()
    print('Top anomalous observations (require individual review):')
    print(extremes.head(20).to_string(index=False))
    print()
    print('ACTION: Review each flagged observation before Phase 2 analysis.')
    print('Do not drop automatically — some may be genuine (COVID crash, etc.).')
else:
    print('No anomalous returns found above the threshold.')

# Coverage check for extended start date (D-020 diagnostic)
print()
print('Coverage by candidate study-start year:')
for candidate_start in ['2005', '2007', '2010']:
    sub = dr.loc[candidate_start:]
    coverage = (sub.notna().sum(axis=1) / dr.shape[1]).mean()
    n_months = sub.resample('MS').first().shape[0]
    print(f'  Start {candidate_start}: {coverage:.1%} avg daily ticker coverage, {n_months} months')

In [ ]:
# ── Return distribution overview ──────────────────────────────────────────────
flat_returns = dr.stack().dropna()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Daily Return Distribution — All Tickers, Full Study Window', fontsize=12)

axes[0].hist(flat_returns, bins=200, color='steelblue', alpha=0.7, edgecolor='none')
axes[0].set_xlim(-0.15, 0.15)
axes[0].set_xlabel('Daily Return')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Histogram (±15% window)')

# Rolling average daily cross-sectional return count
daily_count = dr.notna().sum(axis=1)
axes[1].plot(daily_count.index, daily_count, linewidth=0.8, color='steelblue')
axes[1].set_xlabel('Date')
axes[1].set_ylabel('# Tickers with Return Data')
axes[1].set_title('Daily Universe Coverage')
axes[1].axhline(daily_count.median(), color='red', linestyle='--', alpha=0.5,
                label=f'Median: {daily_count.median():.0f}')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.savefig(DATA_DIR / 'fig_00_data_inspection.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved.')

---

## Step 3 — Universe and FF49 Classification Validation

### Data pipeline

1. Fetch current S&P 500 tickers from Wikipedia
2. Retrieve SIC codes from SEC EDGAR (public API)
3. Map SIC → FF49 industry using the Fama-French crosswalk
4. Fetch daily adjusted prices via yfinance
5. Validate classification coverage and flag anomalies

### Why FF49, not GICS?

GICS (the S&P/MSCI classification available from yfinance metadata) is structurally incompatible with the CLMX methodology. The CLMX paper explicitly uses Fama-French 49 industries (not 48 — the paper text confirms FF49). Substituting GICS would make the decomposition non-comparable to the literature benchmark.

### SIC snapshot metadata requirement

SIC codes are retrieved once from EDGAR and cached. The retrieval date, source endpoint, and mapping version must be recorded — SIC assignments can change as companies amend their filings. Results from different retrieval dates are not guaranteed to be identical.

### Survivorship filter note

This step retrieves *current* S&P 500 membership. Current members projected backward are not equivalent to historical point-in-time membership. This is a documented deviation from CLMX (which used the full CRSP universe with point-in-time constituents). Survivorship bias implications are assessed in the Phase 1 validation notebook (`01_data_validation.ipynb`) and summarized in Step 11.

In [ ]:
# ── 3a: FF49 SIC crosswalk ────────────────────────────────────────────────────
# Functions are defined in Step 12. Load them first if running out of order.
# For sequential execution: Step 12 functions are available after the full
# notebook is loaded (restart kernel, run all).

FF49_CACHE = DATA_DIR / 'ff49_sic_map.csv'

# Inline implementation (Step 12 wraps this in fetch_ff49_crosswalk())
if FF49_CACHE.exists():
    print(f'Loading FF49 crosswalk from cache: {FF49_CACHE}')
    ff49 = pd.read_csv(FF49_CACHE)
else:
    url = 'https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/Siccodes49.zip'
    print('Fetching FF49 SIC crosswalk from French Data Library...')
    resp = requests.get(url, timeout=30)
    resp.raise_for_status()
    with zipfile.ZipFile(io.BytesIO(resp.content)) as zf:
        raw = zf.read(zf.namelist()[0]).decode('latin-1')
    rows = []
    cur_num, cur_name = None, None
    for line in raw.splitlines():
        parts = line.split()
        if len(parts) >= 2 and parts[0].isdigit() and parts[1].isalpha() and len(parts[0]) <= 2:
            cur_num, cur_name = int(parts[0]), parts[1]
        elif len(parts) == 2 and all(p.isdigit() for p in parts):
            try:
                rows.append({'sic_lo': int(parts[0]), 'sic_hi': int(parts[1]),
                             'industry_num': cur_num, 'industry_name': cur_name})
            except (ValueError, TypeError):
                pass
    ff49 = pd.DataFrame(rows)
    ff49.to_csv(FF49_CACHE, index=False)
    print(f'FF49 crosswalk saved ({len(ff49)} SIC ranges, {ff49["industry_num"].nunique()} industries)')

print(f'FF49 industries in crosswalk: {ff49["industry_num"].nunique()}')
print(ff49.head(8))

In [ ]:
# ── 3b: S&P 500 tickers + EDGAR SIC codes ────────────────────────────────────
import datetime

EDGAR_CACHE = DATA_DIR / 'sp500_sic_codes.csv'

# SIC snapshot metadata — required for reproducibility
SIC_SNAPSHOT_META = {
    'retrieval_date': None,   # set at fetch time
    'source': 'SEC EDGAR /submissions/ API',
    'endpoint': 'https://data.sec.gov/submissions/CIK{cik}.json',
    'ticker_index': 'https://www.sec.gov/files/company_tickers.json',
    'mapping_version': 'FF49 from Siccodes49.zip (Kenneth French Data Library)',
}

# Current S&P 500 tickers from Wikipedia
sp500_tickers = pd.read_html(
    'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
)[0]['Symbol'].str.replace('.', '-', regex=False).tolist()
print(f'S&P 500 constituent list: {datetime.date.today()} ({len(sp500_tickers)} tickers)')

if EDGAR_CACHE.exists():
    print(f'Loading SIC codes from cache: {EDGAR_CACHE}')
    sic_df = pd.read_csv(EDGAR_CACHE, dtype={'sic': str})
    SIC_SNAPSHOT_META['retrieval_date'] = 'cached (see file mtime)'
else:
    headers = {'User-Agent': 'Project Parallax research mattnolan.archive@gmail.com'}
    r = requests.get(SIC_SNAPSHOT_META['ticker_index'], headers=headers)
    r.raise_for_status()
    ticker_to_cik = {v['ticker'].upper(): str(v['cik_str']).zfill(10) for v in r.json().values()}
    print(f'EDGAR index: {len(ticker_to_cik)} tickers')

    records = []
    for i, ticker in enumerate(sp500_tickers):
        cik = ticker_to_cik.get(ticker.upper().replace('-', '.')) or ticker_to_cik.get(ticker.upper())
        if cik is None:
            records.append({'ticker': ticker, 'cik': None, 'sic': None, 'company_name': None})
            continue
        try:
            resp = requests.get(f"https://data.sec.gov/submissions/CIK{cik}.json",
                                headers=headers, timeout=15)
            resp.raise_for_status()
            d = resp.json()
            records.append({'ticker': ticker, 'cik': cik,
                            'sic': d.get('sic'), 'company_name': d.get('name')})
        except Exception:
            records.append({'ticker': ticker, 'cik': cik, 'sic': None, 'company_name': None})
        time.sleep(0.12)
        if (i + 1) % 50 == 0:
            print(f'  {i+1}/{len(sp500_tickers)} tickers')

    SIC_SNAPSHOT_META['retrieval_date'] = str(datetime.date.today())
    sic_df = pd.DataFrame(records)
    sic_df.to_csv(EDGAR_CACHE, index=False)
    print(f'SIC codes: {sic_df["sic"].notna().sum()}/{len(sic_df)} matched')

print(f'\nSIC snapshot metadata: {SIC_SNAPSHOT_META}')

In [ ]:
# ── 3c: Map SIC → FF49 and validate coverage ──────────────────────────────────
results = []
for _, row in sic_df.iterrows():
    if pd.isna(row['sic']):
        ind_num, ind_name = 49, 'Other'
    else:
        try:
            sic_int = int(str(row['sic']).replace('.0', ''))
            mask = (ff49['sic_lo'] <= sic_int) & (sic_int <= ff49['sic_hi'])
            matches = ff49[mask]
            if matches.empty:
                ind_num, ind_name = 49, 'Other'
            else:
                ind_num  = int(matches.iloc[0]['industry_num'])
                ind_name = matches.iloc[0]['industry_name']
        except (ValueError, TypeError):
            ind_num, ind_name = 49, 'Other'
    results.append({'ticker': row['ticker'], 'sic': row['sic'],
                    'industry_num': ind_num, 'industry_name': ind_name})

ticker_industry = pd.DataFrame(results).set_index('ticker')

# Coverage validation
n_total    = len(ticker_industry)
n_sic_ok   = sic_df['sic'].notna().sum()
n_other    = (ticker_industry['industry_num'] == 49).sum()
n_placed   = n_total - n_other

print('═' * 55)
print('FF49 Classification Coverage')
print('═' * 55)
print(f'Total tickers       : {n_total}')
print(f'SIC code found      : {n_sic_ok} ({n_sic_ok/n_total:.1%})')
print(f'Placed in FF49 1-48 : {n_placed} ({n_placed/n_total:.1%})')
print(f'Assigned to Other   : {n_other} ({n_other/n_total:.1%})')
print(f'Unique industries   : {ticker_industry["industry_num"].nunique()}')
print()
print('Industry composition (top 15 by count):')
print(
    ticker_industry.groupby(['industry_num', 'industry_name'])
    .size().sort_values(ascending=False).head(15).to_string()
)

# Flag modern-company SIC anomalies worth reviewing
print()
print('Tickers in Other (industry 49) — review for SIC anomalies:')
print(ticker_industry[ticker_industry['industry_num'] == 49].head(20).to_string())

In [ ]:
# ── 3d: Fetch daily prices ────────────────────────────────────────────────────
PRICES_CACHE = DATA_DIR / 'sp500_daily_prices.parquet'

if PRICES_CACHE.exists():
    print(f'Loading prices from cache: {PRICES_CACHE}')
    prices = pd.read_parquet(PRICES_CACHE)
else:
    print(f'Fetching daily adjusted prices for {len(sp500_tickers)} tickers...')
    prices = yf.download(sp500_tickers, start=STUDY_START, end=STUDY_END,
                         auto_adjust=True, progress=False)['Close']
    prices.to_parquet(PRICES_CACHE)
    print(f'Prices saved: {prices.shape}')

# Daily simple returns
raw_returns = prices.pct_change().iloc[1:]

# Restrict to tickers with industry assignments
classified = list(ticker_industry.index)
available  = [t for t in classified if t in raw_returns.columns]
daily_returns = raw_returns[available]

print(f'Price shape  : {prices.shape}')
print(f'Return shape : {daily_returns.shape} (after restricting to classified tickers)')
print(f'Date range   : {daily_returns.index[0].date()} → {daily_returns.index[-1].date()}')

# Step 2 inspection cells can now be run (reference daily_returns above)

---

## Step 4 — Weight Construction

The CLMX decomposition is value-weighted. Weights are computed from prior-month-end market cap and held fixed for the entire month.

**Market cap approximation:** We use price × current shares outstanding. Shares outstanding change slowly relative to daily price, so this approximation is reasonable for weight computation. It will understate weight shifts from significant buyback or issuance events — documented in Step 11.

**Pre-computed weight matrix:** Rather than computing weights inside the decomposition loop, we build a date×ticker weight matrix once and pass it as input. This makes the decomposition engine independent of market-cap data.

**Weight diagnostics to verify:**
- Weights sum to 1.0 for each month
- Largest constituents are plausible (Apple, Microsoft, etc. at the top)
- Month-over-month weight changes are smooth (no sudden composition jumps)

In [ ]:
# ── 4a: Shares outstanding (static proxy) ────────────────────────────────────
SHARES_CACHE = DATA_DIR / 'sp500_shares_outstanding.csv'

if SHARES_CACHE.exists():
    shares_raw = pd.read_csv(SHARES_CACHE, index_col=0).squeeze()
else:
    print(f'Fetching shares outstanding for {len(available)} tickers...')
    shares_dict = {}
    for i, ticker in enumerate(available):
        try:
            shares_dict[ticker] = getattr(yf.Ticker(ticker).fast_info, 'shares', None)
        except Exception:
            shares_dict[ticker] = None
        if (i + 1) % 50 == 0:
            print(f'  {i+1}/{len(available)}')
        time.sleep(0.05)
    shares_raw = pd.Series(shares_dict, name='shares_outstanding')
    shares_raw.to_csv(SHARES_CACHE)
    print(f'Shares retrieved: {shares_raw.notna().sum()}/{len(shares_raw)}')

shares = shares_raw.reindex(available).fillna(shares_raw.median())
n_fallback = shares_raw.reindex(available).isna().sum()
print(f'Tickers using median-fallback shares: {n_fallback}')
print(f'Median shares outstanding: {shares.median():,.0f}')

In [ ]:
# ── 4b: Daily market cap proxy ────────────────────────────────────────────────
approx_mktcap = prices[available].mul(shares, axis='columns')
print(f'Approximate market cap matrix: {approx_mktcap.shape}')

In [ ]:
# ── 4c: Monthly weight computation (shown inline — function in Step 12) ───────
# Compute beginning-of-month VW weights from prior-month closing market cap.

def _compute_month_weights(approx_mktcap, year, month):
    """Inline for Step 4 inspection. See fetch_monthly_weights() in Step 12."""
    bom = pd.Timestamp(year=year, month=month, day=1)
    prior_end = bom - pd.offsets.MonthBegin(1)
    prior_data = approx_mktcap[
        (approx_mktcap.index >= prior_end - pd.offsets.MonthEnd(1)) &
        (approx_mktcap.index < bom)
    ]
    last_mktcap = prior_data.iloc[-1] if not prior_data.empty else approx_mktcap[
        (approx_mktcap.index.year == year) & (approx_mktcap.index.month == month)
    ].iloc[0]
    total = last_mktcap.sum()
    return last_mktcap / total if total > 0 else pd.Series(0.0, index=last_mktcap.index)

# Inspect January 2020 weights
w_jan20 = _compute_month_weights(approx_mktcap, 2020, 1)
print('January 2020 weights — top 10 constituents:')
print(w_jan20.sort_values(ascending=False).head(10).map('{:.3%}'.format).to_string())
print(f'\nWeight sum: {w_jan20.sum():.6f}  (should be 1.0)')

In [ ]:
# ── 4d: Weight diagnostics — stability over time ──────────────────────────────
# Sample every January to see how the top constituent's weight evolves.

years = range(2010, 2025)
top_ticker_shares = []
for yr in years:
    try:
        w = _compute_month_weights(approx_mktcap, yr, 1)
        top = w.idxmax()
        top_ticker_shares.append({'year': yr, 'top_ticker': top, 'weight': w[top]})
    except Exception:
        pass

diag = pd.DataFrame(top_ticker_shares)
print('Largest constituent by year (January weight):')
print(diag.assign(weight=lambda x: x['weight'].map('{:.2%}'.format)).to_string(index=False))
print()
print('Expected: Largest weight is a mega-cap (Apple, Microsoft, Exxon pre-2015, etc.).')
print('Unexpected: Weight > 10% in any single year would warrant investigation.')

---

## Step 5 — Manual Decomposition: MKT Component Only

**Reference month: March 2020** — COVID crash. This month should produce an elevated MKT component because all stocks moved together in response to the same shock.

Show each step of the computation explicitly:
- Select the daily return matrix for the month
- Apply prior-month-end weights
- Compute the daily VW market return μ_d
- Sum μ_d² to get MKT_t

Do not proceed to Steps 6–7 until this output looks correct: daily market returns should show large negative moves in mid-March 2020.

In [ ]:
# ── Step 5: MKT component for March 2020 ─────────────────────────────────────
EX_YEAR, EX_MONTH = 2020, 3

# 5a: Select this month's daily returns
mask_ex = (daily_returns.index.year == EX_YEAR) & (daily_returns.index.month == EX_MONTH)
R_ex = daily_returns[mask_ex].copy()
print(f'March 2020: {len(R_ex)} trading days, {R_ex.shape[1]} tickers before filtering')

# 5b: Prior-month weights
W_ex = _compute_month_weights(approx_mktcap, EX_YEAR, EX_MONTH)
W_ex = W_ex.reindex(R_ex.columns).fillna(0)

# 5c: Apply D-017 (min 10 valid trading days)
# Drop tickers with any NaN this month (complete-month rule for this manual step)
valid_ex = R_ex.columns[R_ex.notna().all()]
R_ex = R_ex[valid_ex]
W_ex = W_ex[valid_ex]
W_ex = W_ex / W_ex.sum()   # re-normalize
print(f'Tickers with complete return history this month: {len(valid_ex)}')

# 5d: Daily VW market return
mu_ex = R_ex.mul(W_ex, axis='columns').sum(axis='columns')

print('\nDaily VW market return, March 2020:')
print(mu_ex.round(4).to_string())

# 5e: MKT component
MKT_ex = (mu_ex**2).sum()
print(f'\nMKT_t = Σ μ_d² = {MKT_ex:.6f}')
print(f'Annualized MKT vol (×12): {np.sqrt(MKT_ex * 12):.2%}')
print()
print('Expected: Large negative returns in the 9–16 March window.')
print('Typical S&P 500 fell ~30% from Feb peak. Daily moves of 5–10% expected.')

---

## Step 6 — Manual Decomposition: IND Component Only

Continuing with March 2020. For each FF49 industry:
- Compute the VW industry return r_{j,d}
- Subtract the market return: η_{j,d} = r_{j,d} - μ_d
- Weight by industry market share: W_j
- Sum η_{j,d}² across days and industries

In a crisis month, IND should be smaller than MKT because all industries moved roughly with the market — little orthogonal industry differentiation.

In [ ]:
# ── Step 6: IND component for March 2020 ─────────────────────────────────────
IND_ex = 0.0
ind_detail = {}

for ind_num in ticker_industry.loc[valid_ex, 'industry_num'].unique():
    members = [
        t for t in ticker_industry[ticker_industry['industry_num'] == ind_num].index
        if t in R_ex.columns
    ]
    if not members:
        continue

    W_j  = W_ex[members].sum()
    if W_j == 0:
        continue
    w_ij = W_ex[members] / W_j
    r_j  = R_ex[members].mul(w_ij, axis='columns').sum(axis='columns')
    eta  = r_j - mu_ex

    ind_contrib = W_j * (eta**2).sum()
    IND_ex += ind_contrib

    ind_name = ticker_industry.loc[members[0], 'industry_name']
    ind_detail[ind_num] = {
        'name': ind_name, 'n_members': len(members),
        'W_j': W_j, 'IND_contrib': ind_contrib, 'r_j': r_j, 'eta': eta,
    }

print('Top 10 industries by IND contribution, March 2020:')
ind_df = pd.DataFrame([
    {'ind_num': k, 'name': v['name'], 'n': v['n_members'],
     'W_j': v['W_j'], 'IND_contrib': v['IND_contrib']}
    for k, v in ind_detail.items()
]).sort_values('IND_contrib', ascending=False)

print(
    ind_df.head(10)
    .assign(W_j=lambda x: x['W_j'].map('{:.2%}'.format),
            IND_contrib=lambda x: x['IND_contrib'].map('{:.6f}'.format))
    .to_string(index=False)
)
print(f'\nIND_t = {IND_ex:.6f}')
print(f'Annualized IND vol (×12): {np.sqrt(IND_ex * 12):.2%}')
print()
print('Expected: IND < MKT in a crisis month (all industries moved together).')

---

## Step 7 — Manual Decomposition: FIRM Component Only

Continuing with March 2020. For each stock:
- Compute the firm residual: ε_{i,d} = r_{i,d} - r_{j,d}
- Weight by industry market share and within-industry weight
- Sum ε_{i,d}² across days, stocks, and industries

In a crisis month, FIRM should be the smallest component — individual company behavior explains little when the entire market crashes together.

In [ ]:
# ── Step 7: FIRM component for March 2020 ────────────────────────────────────
FIRM_ex = 0.0
firm_stock_detail = []

for ind_num, d in ind_detail.items():
    members = [
        t for t in ticker_industry[ticker_industry['industry_num'] == ind_num].index
        if t in R_ex.columns
    ]
    W_j  = W_ex[members].sum()
    w_ij = W_ex[members] / W_j

    for ticker in members:
        eps = R_ex[ticker] - d['r_j']
        eps_sq_sum = (eps**2).sum()
        contrib = W_j * w_ij[ticker] * eps_sq_sum
        FIRM_ex += contrib
        firm_stock_detail.append({
            'ticker': ticker, 'industry': d['name'],
            'w_ij': w_ij[ticker], 'W_j': W_j,
            'eps_sq_sum': eps_sq_sum, 'contrib': contrib,
        })

firm_df = pd.DataFrame(firm_stock_detail).sort_values('contrib', ascending=False)

print('Top 10 individual-firm contributors to FIRM variance, March 2020:')
print(
    firm_df.head(10)[
        ['ticker', 'industry', 'w_ij', 'W_j', 'eps_sq_sum', 'contrib']
    ].assign(
        w_ij=lambda x: x['w_ij'].map('{:.3%}'.format),
        W_j=lambda x: x['W_j'].map('{:.2%}'.format),
        eps_sq_sum=lambda x: x['eps_sq_sum'].map('{:.6f}'.format),
        contrib=lambda x: x['contrib'].map('{:.7f}'.format),
    ).to_string(index=False)
)

total_ex = MKT_ex + IND_ex + FIRM_ex

print()
print('═' * 60)
print(f'March 2020 — Complete Decomposition')
print('═' * 60)
print(f'MKT  : {MKT_ex:.6f}  ({MKT_ex/total_ex:.1%})')
print(f'IND  : {IND_ex:.6f}  ({IND_ex/total_ex:.1%})')
print(f'FIRM : {FIRM_ex:.6f}  ({FIRM_ex/total_ex:.1%})')
print(f'Total: {total_ex:.6f}')
print()
print('Expected: MKT dominant (>50%), FIRM lowest share. Crisis = common movement.')
print('If FIRM > MKT in March 2020, revisit industry classification pipeline.')

---

## Step 8 — Full-Period Monthly Accumulation

Apply the decomposition across all months in the study window.

### Decision D-018 — Open: Stock Inclusion Rule

Two options for determining which stocks enter each month's decomposition:

**Option A — Valid-Day Inclusion:** Include any stock with ≥ 10 valid daily return observations in the month (Decision D-017 threshold). A stock's daily returns may be missing on some days without excluding it.

**Option B — Complete-Month Completeness:** Include only stocks with valid returns on *every* trading day. A single missing day excludes the stock from that month.

These options have a structural difference: under Option A, the effective VW market portfolio changes composition day-by-day within a month (a stock may be present on some days but not others). Under Option B, the portfolio is fixed for the month. This affects VW-market-return consistency across days.

**D-018 is unresolved.** Both options are run below. The decisive check is a single overlay chart: do the FIRM variance series materially diverge? If the difference is economically small, Option B (simpler, cleaner) is preferred. Decision will be registered after reviewing the chart.

**D-017** (min 10 valid days) applies under both options — a stock with fewer than 10 valid days is excluded regardless.

In [ ]:
# ── D-018 stock-set builder ───────────────────────────────────────────────────
def build_monthly_stock_set(
    daily_returns: pd.DataFrame,
    year: int,
    month: int,
    min_valid_days: int = 10,
    missing_treatment: str = 'complete_month',   # 'complete_month' | 'valid_days'
) -> pd.Index:
    """Return the set of valid tickers for a given month under a specified D-018 option.

    Parameters
    ----------
    missing_treatment : 'complete_month' (Option B) | 'valid_days' (Option A)
        'complete_month': include stocks valid on every trading day.
        'valid_days':     include stocks valid on >= min_valid_days days.
    """
    mask = (daily_returns.index.year == year) & (daily_returns.index.month == month)
    R = daily_returns[mask]
    if missing_treatment == 'complete_month':
        valid = R.columns[R.notna().all()]
    else:  # 'valid_days'
        valid = R.columns[R.notna().sum() >= min_valid_days]
    return valid


# ── Monthly decomposition engine (calls Step 12 helper functions) ──────────────
def decompose_month(
    daily_returns, approx_mktcap, ticker_industry,
    year, month,
    min_valid_days=10,
    missing_treatment='complete_month',
):
    """Single-month CLMX decomposition. Returns dict or None."""
    mask = (daily_returns.index.year == year) & (daily_returns.index.month == month)
    R_all = daily_returns[mask]
    if len(R_all) < min_valid_days:
        return None

    valid = build_monthly_stock_set(daily_returns, year, month,
                                    min_valid_days, missing_treatment)
    R = R_all[valid]
    if R.empty:
        return None

    W = _compute_month_weights(approx_mktcap, year, month)
    W = W.reindex(valid).fillna(0)
    total_w = W.sum()
    if total_w == 0:
        return None
    W = W / total_w

    mu_d = R.mul(W, axis='columns').sum(axis='columns')
    MKT  = (mu_d**2).sum()
    IND  = 0.0
    FIRM = 0.0
    industries_seen = set()

    for ind_num in ticker_industry.loc[valid, 'industry_num'].unique():
        members = [
            t for t in ticker_industry[ticker_industry['industry_num'] == ind_num].index
            if t in R.columns
        ]
        if not members:
            continue
        W_j = W[members].sum()
        if W_j == 0:
            continue
        w_ij = W[members] / W_j
        r_j  = R[members].mul(w_ij, axis='columns').sum(axis='columns')
        eta  = r_j - mu_d
        IND += W_j * (eta**2).sum()
        for ticker in members:
            eps = R[ticker] - r_j
            FIRM += W_j * w_ij[ticker] * (eps**2).sum()
        industries_seen.add(ind_num)

    return {
        'year': year, 'month': month,
        'MKT': MKT, 'IND': IND, 'FIRM': FIRM,
        'n_stocks': len(valid), 'n_industries': len(industries_seen),
        'n_trading_days': len(R),
    }

In [ ]:
# ── Full-period loop ───────────────────────────────────────────────────────────
RESULTS_CACHE_B = DATA_DIR / 'clmx_monthly_decomposition.csv'
RESULTS_CACHE_A = DATA_DIR / 'clmx_monthly_decomposition_opt_a.csv'

study_months = pd.date_range(start=STUDY_START, end=STUDY_END, freq='MS')

def run_decomposition(cache_path, missing_treatment):
    if cache_path.exists():
        print(f'Loading from cache: {cache_path}')
        return pd.read_csv(cache_path)
    print(f'Running decomposition (missing_treatment={missing_treatment!r})...')
    records = []
    for dt in study_months:
        r = decompose_month(
            daily_returns, approx_mktcap, ticker_industry,
            year=dt.year, month=dt.month,
            missing_treatment=missing_treatment,
        )
        if r is not None:
            records.append(r)
        if dt.month == 1:
            print(f'  Completed {dt.year}')
    df = pd.DataFrame(records)
    df.to_csv(cache_path, index=False)
    print(f'Saved {len(df)} months to {cache_path}')
    return df

# Option B (complete_month) — primary
monthly_B = run_decomposition(RESULTS_CACHE_B, 'complete_month')
# Option A (valid_days) — comparison
monthly_A = run_decomposition(RESULTS_CACHE_A, 'valid_days')

for df, label in [(monthly_B, 'Option B (complete_month)'),
                   (monthly_A, 'Option A (valid_days)')]:
    df['date'] = pd.to_datetime(df[['year', 'month']].assign(day=1))
    df.set_index('date', inplace=True)
    df.sort_index(inplace=True)
    print(f'{label}: {len(df)} months, avg {df["n_stocks"].mean():.0f} stocks/month')

In [ ]:
# ── D-018 comparison chart ────────────────────────────────────────────────────
# The decisive visual check: do the two options materially diverge?
# If the overlay lines are nearly identical, D-018 has little practical consequence.

fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
fig.suptitle('D-018 Comparison: Option A (valid_days) vs Option B (complete_month)',
             fontsize=12, fontweight='bold')

# FIRM variance series
ax = axes[0]
ax.plot(monthly_A.index, np.sqrt(monthly_A['FIRM'].rolling(12).sum()) * 100,
        label='Option A — valid_days', color='#d01c8b', linewidth=1.5)
ax.plot(monthly_B.index, np.sqrt(monthly_B['FIRM'].rolling(12).sum()) * 100,
        label='Option B — complete_month', color='#2166ac', linewidth=1.5, linestyle='--')
ax.set_ylabel('FIRM Annualized Vol (%, 12m rolling)')
ax.legend(fontsize=10)
ax.grid(alpha=0.3, linestyle='--')

# n_stocks contributing
ax2 = axes[1]
ax2.plot(monthly_A.index, monthly_A['n_stocks'], label='Option A', color='#d01c8b')
ax2.plot(monthly_B.index, monthly_B['n_stocks'], label='Option B', color='#2166ac', linestyle='--')
ax2.set_ylabel('# Stocks Contributing')
ax2.set_xlabel('Date')
ax2.legend(fontsize=10)
ax2.grid(alpha=0.3, linestyle='--')

plt.tight_layout()
plt.savefig(DATA_DIR / 'fig_d018_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# Numeric summary
print('D-018 FIRM variance difference (Option A minus Option B):')
diff = monthly_A['FIRM'].reindex(monthly_B.index) - monthly_B['FIRM']
rel_diff = (diff / monthly_B['FIRM']).abs()
print(f'  Median absolute relative difference: {rel_diff.median():.2%}')
print(f'  Max absolute relative difference:    {rel_diff.max():.2%}')
print()
print('D-018 decision criteria:')
print('  < 2% median relative diff → Options are economically equivalent, prefer B')
print('  > 5% median relative diff → Material; document and explain before proceeding')

In [ ]:
# ── Annual aggregation (Option B / primary) ────────────────────────────────────
# Primary series uses Option B pending D-018 resolution.
monthly_results = monthly_B.copy()

annual = monthly_results[['MKT', 'IND', 'FIRM']].resample('YE').sum()
annual.index = annual.index.year
annual['total']      = annual.sum(axis=1)
annual['MKT_share']  = annual['MKT']  / annual['total']
annual['IND_share']  = annual['IND']  / annual['total']
annual['FIRM_share'] = annual['FIRM'] / annual['total']

print('Annual variance shares:')
print(annual[['MKT_share', 'IND_share', 'FIRM_share']].round(3).to_string())

---

## Step 9 — Reconciliation and Sanity Checks

Verify that the decomposition is internally consistent before drawing any conclusions.

**Checks:**
1. MKT + IND + FIRM ≈ VW-average individual stock monthly variance (reconciliation)
2. All components are non-negative (assertion)
3. Per-month n_stocks_contributing shows no suspicious collapses
4. Outlier months are economically identifiable (COVID, GFC, etc.)

A median reconciliation gap > 5% would indicate a computational error that must be diagnosed before any results are interpreted.

In [ ]:
# ── 9a: Non-negativity assertion ──────────────────────────────────────────────
for col in ['MKT', 'IND', 'FIRM']:
    n_neg = (monthly_results[col] < 0).sum()
    assert n_neg == 0, f'{col} has {n_neg} negative values — computation error'
    print(f'{col}: all {len(monthly_results)} months non-negative ✓')

In [ ]:
# ── 9b: Reconciliation vs. VW-average individual stock variance ───────────────
RECON_CACHE = DATA_DIR / 'clmx_reconciliation.csv'

if not RECON_CACHE.exists():
    recon_rows = []
    for dt, row in monthly_results.iterrows():
        yr, mo = dt.year, dt.month
        mask = (daily_returns.index.year == yr) & (daily_returns.index.month == mo)
        R_r = daily_returns[mask]
        valid_r = R_r.columns[R_r.notna().all()]
        R_r = R_r[valid_r]
        W_r = _compute_month_weights(approx_mktcap, yr, mo)
        W_r = W_r.reindex(valid_r).fillna(0)
        if W_r.sum() == 0 or R_r.empty:
            continue
        W_r = W_r / W_r.sum()
        vw_total = (R_r**2).sum().mul(W_r).sum()
        comp_sum = row['MKT'] + row['IND'] + row['FIRM']
        recon_rows.append({'date': dt, 'VW_total': vw_total, 'comp_sum': comp_sum})
    recon_df = pd.DataFrame(recon_rows).set_index('date')
    recon_df.to_csv(RECON_CACHE)
else:
    recon_df = pd.read_csv(RECON_CACHE, index_col=0, parse_dates=True)

recon_df['diff_pct'] = ((recon_df['VW_total'] - recon_df['comp_sum']) / recon_df['VW_total']).abs()

print('Reconciliation: MKT + IND + FIRM vs. VW-avg individual stock variance')
print(f'Median absolute gap: {recon_df["diff_pct"].median():.2%}')
print(f'Max absolute gap:    {recon_df["diff_pct"].max():.2%}')
print(f'Months with > 5% gap: {(recon_df["diff_pct"] > 0.05).sum()}')
print()
print('Differences arise from cross-product terms in the decomposition identity.')
print('< 2% median gap is expected and acceptable.')

In [ ]:
# ── 9c: Per-month n_stocks and outlier table ──────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
fig.suptitle('Sanity Checks — Universe Coverage and Outlier Months', fontsize=12)

axes[0].plot(monthly_results.index, monthly_results['n_stocks'],
             color='steelblue', linewidth=1)
axes[0].set_ylabel('# Stocks Contributing')
axes[0].grid(alpha=0.3, linestyle='--')
axes[0].set_title('Stocks per Month (should be stable, ~400–500)')

axes[1].plot(recon_df.index, recon_df['diff_pct'] * 100,
             color='firebrick', linewidth=1)
axes[1].axhline(5, color='orange', linestyle='--', alpha=0.7, label='5% threshold')
axes[1].set_ylabel('Reconciliation Gap (%)')
axes[1].set_xlabel('Date')
axes[1].legend(fontsize=9)
axes[1].grid(alpha=0.3, linestyle='--')

plt.tight_layout()
plt.savefig(DATA_DIR / 'fig_sanity_checks.png', dpi=150, bbox_inches='tight')
plt.show()

# Outlier months
total_var = monthly_results['MKT'] + monthly_results['IND'] + monthly_results['FIRM']
outlier_months = total_var.nlargest(10)
print('Top 10 months by total variance (should include COVID, GFC, rate-hike periods):')
for dt, val in outlier_months.items():
    mkt_share = monthly_results.loc[dt, 'MKT'] / val
    firm_share = monthly_results.loc[dt, 'FIRM'] / val
    print(f'  {dt.strftime("%Y-%m")}: total={val:.5f}  MKT={mkt_share:.1%}  FIRM={firm_share:.1%}')

---

## Step 10 — Benchmark Comparison to CLMX (2022)

Directional comparison against CLMX (2022) NBER WP 29916, Figures 2–4.

**We do not attempt exact numerical replication.** Our universe (S&P 500 via yfinance, current members) is structurally different from the full CRSP universe used in CLMX. The comparison is directional and qualitative.

**Expected pattern from CLMX (2022) for the post-2001 period:**

| Feature | CLMX (2022) post-2001 finding |
|---|---|
| FIRM variance share trend | No persistent secular increase (contrast with 1962–1997) |
| MKT component in crisis months | Sharp spikes during GFC, COVID |
| FIRM share in crisis vs. calm | Lower during crises, higher in expansions |
| Overall vol levels | Higher than pre-2001 averages, but FIRM share lower |

Our estimates will show **lower absolute volatility levels** than CLMX because the S&P 500 large-cap universe is materially less volatile than the full CRSP universe (which includes thousands of smaller, more volatile stocks).

In [ ]:
# ── Time series of annualized volatility components ───────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)
fig.suptitle(
    'CLMX Variance Decomposition — S&P 500 (Public Data Approximation)\n'
    'Annualized Volatility by Component | Compare to CLMX (2022) Figures 2–3',
    fontsize=12, fontweight='bold'
)

colors  = {'MKT': '#2166ac', 'IND': '#4dac26', 'FIRM': '#d01c8b'}
clabels = {'MKT': 'Market', 'IND': 'Industry', 'FIRM': 'Firm-Specific'}

for ax, comp in zip(axes, ['MKT', 'IND', 'FIRM']):
    roll_vol = np.sqrt(monthly_results[comp].rolling(12).sum()) * 100
    ax.plot(roll_vol.index, roll_vol, color=colors[comp], linewidth=1.5)
    ax.fill_between(roll_vol.index, roll_vol, alpha=0.15, color=colors[comp])
    ax.set_ylabel(f'{clabels[comp]}\nVol (% ann.)', fontsize=10)
    ax.grid(alpha=0.3, linestyle='--')

events = [('2020-03-01', 'COVID'), ('2022-01-01', 'Rate hikes')]
for date_str, lbl in events:
    for ax in axes:
        ax.axvline(pd.Timestamp(date_str), color='gray', linestyle=':', alpha=0.7, linewidth=0.8)

axes[-1].set_xlabel('Date')
plt.tight_layout()
plt.savefig(DATA_DIR / 'fig_01_clmx_components_time_series.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Variance share chart — compare to CLMX (2022) Figure 4 ───────────────────
fig, ax = plt.subplots(figsize=(12, 5))

rolling_total = monthly_results[['MKT', 'IND', 'FIRM']].rolling(24).sum()
rolling_shares = rolling_total.div(rolling_total.sum(axis=1), axis=0)

ax.stackplot(
    rolling_shares.index,
    rolling_shares['FIRM'] * 100,
    rolling_shares['IND'] * 100,
    rolling_shares['MKT'] * 100,
    labels=['Firm-Specific', 'Industry', 'Market'],
    colors=['#d01c8b', '#4dac26', '#2166ac'], alpha=0.8,
)
ax.set_ylabel('Variance Share (%)')
ax.set_xlabel('Date')
ax.set_title('Variance Shares: MKT / IND / FIRM — 24-Month Rolling | Compare to CLMX (2022) Fig. 4')
ax.legend(loc='upper left', fontsize=10)
ax.set_ylim(0, 100)
ax.grid(alpha=0.3, linestyle='--', axis='y')
plt.tight_layout()
plt.savefig(DATA_DIR / 'fig_02_variance_shares.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Sub-period summary + directional consistency check ────────────────────────
subperiods = {
    '2010–2014': ('2010-01', '2014-12'),
    '2015–2019': ('2015-01', '2019-12'),
    '2020–2021': ('2020-01', '2021-12'),
    '2022–2024': ('2022-01', '2024-12'),
    'Full 2010–2024': (STUDY_START[:7], STUDY_END[:7]),
}

rows = []
for lbl, (s, e) in subperiods.items():
    sub = monthly_results.loc[s:e, ['MKT', 'IND', 'FIRM']]
    if sub.empty:
        continue
    avg_ann = sub.mean() * 12
    avg_vol = np.sqrt(avg_ann) * 100
    total   = avg_ann.sum()
    rows.append({'Period': lbl,
                 'MKT Vol': f"{avg_vol['MKT']:.1f}%",
                 'IND Vol': f"{avg_vol['IND']:.1f}%",
                 'FIRM Vol': f"{avg_vol['FIRM']:.1f}%",
                 'FIRM Share': f"{avg_ann['FIRM']/total:.1%}",
                 'N': len(sub)})

print('Sub-period Summary (annualized volatilities)')
print('=' * 70)
print(pd.DataFrame(rows).to_string(index=False))
print()

# Crisis vs. calm directional check
monthly_idx = monthly_results.copy()
monthly_idx.index = monthly_idx.index.strftime('%Y-%m')

for label, months in [
    ('Crisis', ['2020-03', '2020-04', '2022-06', '2022-09']),
    ('Calm',   ['2017-06', '2017-07', '2019-06', '2019-07']),
]:
    sel = monthly_idx.loc[[m for m in months if m in monthly_idx.index]]
    if sel.empty:
        continue
    total = sel[['MKT', 'IND', 'FIRM']].sum(axis=1)
    shares = sel[['MKT', 'IND', 'FIRM']].div(total, axis=0).mean()
    print(f'{label} months avg shares:  '
          f'MKT={shares["MKT"]:.1%}  IND={shares["IND"]:.1%}  FIRM={shares["FIRM"]:.1%}')

print()
print('Expected: MKT share higher in crisis months, FIRM share higher in calm months.')
print('Consistent with CLMX (2022) Figure 4 qualitative pattern.')

---

## Step 11 — Limitations

Every meaningful difference between this implementation and the original CLMX study is documented here, organized by expected impact on inference.

### HIGH Priority — Likely to materially affect interpretation

**Survivorship bias (universe construction)**
We use current S&P 500 membership projected backward. Current constituents are companies that survived — companies removed for poor performance, delisted, or acquired are absent. Survivorship creates two distortions: (1) historical FIRM variance is understated because distressed companies (which tend to have high idiosyncratic volatility) are excluded; (2) historical return levels are overstated. The direction of the bias is clear; the magnitude requires Phase 1 diagnostic work. This is a first-class analytical risk, not a footnote.

**Incomplete historical return coverage**
Some current S&P 500 members did not exist, were not publicly traded, or were not large enough to join the index in 2010. Their historical data begins after the study start, creating partial histories. The decomposition uses whatever data is available, but the effective universe changes over time in a non-random way (companies added later tend to be growth-oriented, recently successful, or from industries underrepresented in 2010).

### MEDIUM Priority — May affect results in specific periods

**Market-cap approximation (static shares outstanding)**
We use current shares × historical prices as the VW weight proxy. Shares outstanding change through buybacks, issuances, and splits. For companies with substantial buyback programs (technology companies, consumer staples), current shares understate historical share counts, biasing historical weights toward current mega-cap concentrations. Effect is largest over long horizons.

**Delisting bias**
When a company is removed from the S&P 500 and its ticker becomes unavailable in yfinance, we lose the trailing return history for that period. If the removal was performance-related, the excluded period likely had elevated firm-specific volatility — another understating mechanism for FIRM variance.

**VW market-return drift**
Our daily VW market return (μ_d) is computed over the available-stock universe only. If the largest stocks have gaps on specific days (data quality issues), μ_d drifts from the true VW index return. This is a computational consistency concern rather than a systematic bias.

### LOWER Priority — Small or hard to quantify

**SIC code drift**
SIC codes are retrieved at a single snapshot date from EDGAR. Companies sometimes update their primary SIC code as they diversify or shift business focus. A technology company that started in hardware may have originally filed under a manufacturing SIC code. We use current SIC codes for the full historical period, misclassifying companies for years when their actual industry focus differed.

**Ticker changes and corporate actions**
Mergers, acquisitions, and spinoffs can create discontinuities in return histories. yfinance handles most routine splits and dividend adjustments, but structural corporate changes may create artificial return spikes or gaps that the anomaly screen in Step 2 is designed to catch.

### CLMX Deviation Summary Table

| Dimension | CLMX (2001/2022) | This Replication | Expected Effect |
|---|---|---|---|
| Universe | Full CRSP (~3,000–8,000 stocks) | S&P 500 current members (~500) | Lower vol levels; understated FIRM |
| Data source | CRSP (institutional, point-in-time) | yfinance (free, current S&P 500 only) | Survivorship bias |
| Market weights | CRSP daily market cap (price × shrout) | Price × current shares (static approx.) | Small weight distortion |
| Study period | 1962–2021 | 2010–2024 | No pre-2001 comparison |
| Missing data rule | CRSP-standard exclusions | All-days-valid per month (Option B) | More aggressive exclusion |
| Min observation rule | Not documented | 10 trading days (D-017) | Documented Parallax design choice |

### What constitutes a successful directional replication

1. Crisis months (COVID 2020, rate-hike 2022) show elevated MKT share — ✓ check Step 10
2. Expansion months show MKT share declining and FIRM share recovering — ✓ check Step 10
3. No obvious secular upward trend in FIRM share over 2010–2024 — consistent with CLMX (2022)
4. Annualized MKT/IND/FIRM volatilities are lower than CLMX's CRSP-based estimates — expected

---

## Step 12 — Reusable Functions

All helper functions are consolidated here. Steps 1–11 showed computations inline to make each decision explicit. These functions package that logic for use in downstream notebooks and modules.

**To use in another notebook:**
```python
import sys; sys.path.insert(0, '../src')
# Or copy the relevant function bodies
```

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Step 12 — Reusable Functions
# All function definitions are consolidated here.
# These are the canonical implementations; Steps 1–11 showed them inline.
# ══════════════════════════════════════════════════════════════════════════════

def fetch_ff49_crosswalk(cache_path: Path = FF49_CACHE) -> pd.DataFrame:
    """Download and parse the Fama-French 49-industry SIC crosswalk.

    Returns DataFrame: sic_lo, sic_hi, industry_num, industry_name.
    Caches to cache_path to avoid repeated downloads.
    Source: https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/Siccodes49.zip
    """
    if cache_path.exists():
        return pd.read_csv(cache_path)
    url = 'https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/Siccodes49.zip'
    resp = requests.get(url, timeout=30)
    resp.raise_for_status()
    with zipfile.ZipFile(io.BytesIO(resp.content)) as zf:
        raw = zf.read(zf.namelist()[0]).decode('latin-1')
    rows, cur_num, cur_name = [], None, None
    for line in raw.splitlines():
        parts = line.split()
        if len(parts) >= 2 and parts[0].isdigit() and parts[1].isalpha() and len(parts[0]) <= 2:
            cur_num, cur_name = int(parts[0]), parts[1]
        elif len(parts) == 2 and all(p.isdigit() for p in parts):
            try:
                rows.append({'sic_lo': int(parts[0]), 'sic_hi': int(parts[1]),
                             'industry_num': cur_num, 'industry_name': cur_name})
            except (ValueError, TypeError):
                pass
    df = pd.DataFrame(rows)
    df.to_csv(cache_path, index=False)
    return df


def sic_to_ff49(sic: int, crosswalk: pd.DataFrame) -> Tuple[int, str]:
    """Map a 4-digit SIC code to FF49 industry. Returns (49, 'Other') if unmatched."""
    mask = (crosswalk['sic_lo'] <= sic) & (sic <= crosswalk['sic_hi'])
    matches = crosswalk[mask]
    if matches.empty:
        return (49, 'Other')
    return (int(matches.iloc[0]['industry_num']), matches.iloc[0]['industry_name'])


def fetch_edgar_sic_codes(
    tickers: list,
    cache_path: Path = EDGAR_CACHE,
    user_agent: str = 'Project Parallax research mattnolan.archive@gmail.com',
) -> pd.DataFrame:
    """Fetch SIC codes for a list of tickers from SEC EDGAR.

    Caches results. EDGAR user-agent policy requires a contact email.
    Rate limit: 0.12s delay per request (~8 req/sec).
    Returns DataFrame: ticker, cik, sic, company_name.
    """
    if cache_path.exists():
        return pd.read_csv(cache_path, dtype={'sic': str})
    headers = {'User-Agent': user_agent}
    r = requests.get('https://www.sec.gov/files/company_tickers.json', headers=headers)
    r.raise_for_status()
    ticker_to_cik = {v['ticker'].upper(): str(v['cik_str']).zfill(10) for v in r.json().values()}
    records = []
    for ticker in tickers:
        cik = ticker_to_cik.get(ticker.upper().replace('-', '.')) or ticker_to_cik.get(ticker.upper())
        if cik is None:
            records.append({'ticker': ticker, 'cik': None, 'sic': None, 'company_name': None})
            continue
        try:
            resp = requests.get(f'https://data.sec.gov/submissions/CIK{cik}.json',
                                headers=headers, timeout=15)
            resp.raise_for_status()
            d = resp.json()
            records.append({'ticker': ticker, 'cik': cik,
                            'sic': d.get('sic'), 'company_name': d.get('name')})
        except Exception:
            records.append({'ticker': ticker, 'cik': cik, 'sic': None, 'company_name': None})
        time.sleep(0.12)
    df = pd.DataFrame(records)
    df.to_csv(cache_path, index=False)
    return df


def build_ticker_industry_map(
    sic_df: pd.DataFrame, ff49: pd.DataFrame
) -> pd.DataFrame:
    """Map each ticker to its Fama-French 49 industry using its SIC code.

    Returns DataFrame indexed by ticker: sic, industry_num, industry_name.
    Unmatched tickers receive industry 49 ('Other').
    """
    results = []
    for _, row in sic_df.iterrows():
        if pd.isna(row['sic']):
            ind_num, ind_name = 49, 'Other'
        else:
            try:
                sic_int = int(str(row['sic']).replace('.0', ''))
                ind_num, ind_name = sic_to_ff49(sic_int, ff49)
            except (ValueError, TypeError):
                ind_num, ind_name = 49, 'Other'
        results.append({'ticker': row['ticker'], 'sic': row['sic'],
                        'industry_num': ind_num, 'industry_name': ind_name})
    return pd.DataFrame(results).set_index('ticker')


def compute_monthly_weights(
    approx_mktcap: pd.DataFrame, year: int, month: int
) -> pd.Series:
    """Compute beginning-of-month VW weights from prior-month closing market cap.

    Returns Series summing to 1.0. Stocks with no prior-month data receive zero weight.
    For the first available month, uses beginning-of-month market cap as fallback.
    """
    bom = pd.Timestamp(year=year, month=month, day=1)
    prior_end = bom - pd.offsets.MonthBegin(1)
    prior_data = approx_mktcap[
        (approx_mktcap.index >= prior_end - pd.offsets.MonthEnd(1)) &
        (approx_mktcap.index < bom)
    ]
    if prior_data.empty:
        cur = approx_mktcap[
            (approx_mktcap.index.year == year) & (approx_mktcap.index.month == month)
        ]
        last_mktcap = cur.iloc[0] if not cur.empty else pd.Series(dtype=float)
    else:
        last_mktcap = prior_data.iloc[-1]
    total = last_mktcap.sum()
    return last_mktcap / total if total > 0 else pd.Series(0.0, index=last_mktcap.index)


def clmx_decompose_month(
    daily_returns: pd.DataFrame,
    approx_mktcap: pd.DataFrame,
    ticker_industry: pd.DataFrame,
    year: int,
    month: int,
    min_valid_days: int = 10,
    missing_treatment: str = 'complete_month',
) -> Optional[Dict]:
    """Compute CLMX MKT/IND/FIRM variance decomposition for a single calendar month.

    Parameters
    ----------
    missing_treatment : 'complete_month' (D-018 Option B) | 'valid_days' (D-018 Option A)
        D-018 is unresolved. Run both and compare via build_monthly_stock_set().
    min_valid_days : int
        D-017: minimum valid trading days per stock per month (Parallax design choice;
        not an original CLMX requirement).

    Returns
    -------
    dict : year, month, MKT, IND, FIRM, n_stocks, n_industries, n_trading_days
    None : if the month has insufficient data.

    Notes
    -----
    Estimator confirmed from CLMX (2022) NBER WP 29916:
    - Monthly variance = sum of raw squared daily return components (not demeaned)
    - Not normalized by trading-day count
    - Value-weighted primary (D-019)

    CLMX FIRM variance is NOT equivalent to FF6-residual variance (Phase 5).
    This function removes market and industry components only.
    """
    mask = (daily_returns.index.year == year) & (daily_returns.index.month == month)
    R_all = daily_returns[mask]
    if len(R_all) < min_valid_days:
        return None

    valid = build_monthly_stock_set(daily_returns, year, month,
                                    min_valid_days, missing_treatment)
    R = R_all[valid]
    if R.empty:
        return None

    W = compute_monthly_weights(approx_mktcap, year, month)
    W = W.reindex(valid).fillna(0)
    total_w = W.sum()
    if total_w == 0:
        return None
    W = W / total_w

    mu_d = R.mul(W, axis='columns').sum(axis='columns')
    MKT  = (mu_d**2).sum()
    IND  = 0.0
    FIRM = 0.0
    industries_seen = set()

    for ind_num in ticker_industry.loc[valid, 'industry_num'].unique():
        members = [
            t for t in ticker_industry[ticker_industry['industry_num'] == ind_num].index
            if t in R.columns
        ]
        if not members:
            continue
        W_j = W[members].sum()
        if W_j == 0:
            continue
        w_ij = W[members] / W_j
        r_j  = R[members].mul(w_ij, axis='columns').sum(axis='columns')
        eta  = r_j - mu_d
        IND += W_j * (eta**2).sum()
        for ticker in members:
            eps = R[ticker] - r_j
            FIRM += W_j * w_ij[ticker] * (eps**2).sum()
        industries_seen.add(ind_num)

    return {
        'year': year, 'month': month,
        'MKT': MKT, 'IND': IND, 'FIRM': FIRM,
        'n_stocks': len(valid),
        'n_industries': len(industries_seen),
        'n_trading_days': len(R),
    }


print('Step 12 functions defined:')
fns = ['fetch_ff49_crosswalk', 'sic_to_ff49', 'fetch_edgar_sic_codes',
       'build_ticker_industry_map', 'compute_monthly_weights',
       'build_monthly_stock_set', 'clmx_decompose_month']
for fn in fns:
    print(f'  ✓ {fn}()')